# Reconstruction depuis un nuage dégradé — **version Google Colab (GPU)**

`Exécution > Modifier le type d'exécution > GPU`, puis glisse
`Code/output/src_pour_colab.zip` dans le panneau **Fichiers**.

**Vérifie la cellule 1.** Si elle affiche `CUDA : False`, tu es sur le processeur de
Colab, qui n'a que 2 cœurs — ton portable est alors **plus rapide**. Dans ce cas, utilise
`Points2Surf_local.ipynb` sur ta machine. Le notebook s'adapte quand même et retombe
automatiquement sur des réglages CPU allégés.

## Le protocole

1. **Étalon de référence** : une forme difficile (donut + boule collée), maillée exactement.
2. **On abîme les données** : on efface une zone du nuage (comme un scanner qui rate une
   face) et on ajoute du bruit.
3. **Deux reconstructions** depuis ce nuage abîmé :
   - la **méthode du projet** — signe par produit scalaire `(P−S)·n(S)`, avec normales ;
   - **mini-Points2Surf** — `|f|` (voisinage local) × `signe` (vue globale), sans normales.
4. **On compare les deux à l'étalon.**

## Pourquoi abîmer le nuage

Points2Surf annonce un gain sur les scans *bruités et partiels*. Sur un nuage propre les
deux méthodes font jeu égal — le produit scalaire est même meilleur, car il est exact
alors que le réseau approxime. C'est là où il manque de la matière que la question se
pose vraiment.

In [ ]:
# 1) Matériel
import torch, sys
print('python :', sys.version.split()[0], '| torch :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', DEVICE)

## Récupérer le code

Le dépôt est **privé**, donc `git clone` ne marche pas ici. Glisse
`Code/output/src_pour_colab.zip` dans le panneau **Fichiers** de Colab (icône dossier à
gauche), puis exécute la cellule.

In [ ]:
# 2) Mettre src/ a disposition, sans git
import os, zipfile, glob, sys

if not os.path.isdir('src'):
    zips = glob.glob('/content/*.zip') + glob.glob('*.zip')
    if zips:
        print('zip trouve :', zips[0])
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall('.')

if not os.path.isdir('src'):
    print('src/ introuvable.')
    print('Glisse Code/output/src_pour_colab.zip dans le panneau Fichiers, puis relance.')
    raise SystemExit(1)

sys.path.insert(0, os.getcwd())
from src.shapes import get_surface
print('import OK  |  cwd =', os.getcwd())

## 1) L'étalon de référence

**La forme.** Pas une sphère : on prend un objet géométriquement pénible.

- `'tore'` — genre 1. Sa caractéristique d'Euler vaut **0** et non 2, ce qui en fait un
  test topologique sévère : une reconstruction qui bouche le trou donnera `χ = 2` et se
  trahira immédiatement. Son tube intérieur est de plus un cauchemar d'axe médian.
- `'tore_sphere'` — union CSG d'un tore et d'une sphère, via le module `csg.py` du
  projet. Genre 1 **plus** une arête vive à l'intersection, là où le gradient est
  discontinu. C'est le cas le plus dur.
- `'globule'` — la quartique de Kuchel, pour comparer avec le reste du projet.

L'étalon est le maillage obtenu par marching cubes sur la **formule exacte**, à haute
résolution. C'est lui, et lui seul, qui sert de vérité dans tout ce qui suit.

In [ ]:
import numpy as np
from skimage import measure
from scipy.spatial import cKDTree

from src.shapes import implicit_torus, implicit_rbc_unit
from src.csg import union, translated
from src.inside_outside import OrientedCloud
from src.invariants import (is_watertight, euler_characteristic, mesh_area,
                            mesh_volume, weld_vertices)
from src.cloud_io import face_normals, vertex_normals, orient_outward, mesh_signed_volume
from src.metrics import chamfer_distance

# ------------------------------------------------------------------
FORME = 'tore_sphere'      # 'tore' | 'tore_sphere' | 'globule'
# ------------------------------------------------------------------

def sphere_field(R=0.55, centre=(0.85, 0.0, 0.0)):
    c = np.asarray(centre, dtype=np.float64).reshape(1, 3)
    def f(P):
        P = np.asarray(P, dtype=np.float64).reshape(-1, 3)
        return np.linalg.norm(P - c, axis=1) - R
    return f

if FORME == 'tore':
    champ, BOX, chi_attendu = (lambda P: implicit_torus(P)), 1.5, 0
elif FORME == 'tore_sphere':
    champ, BOX, chi_attendu = union(implicit_torus, sphere_field()), 1.6, 0
else:
    champ, BOX, chi_attendu = (lambda P: implicit_rbc_unit(P)), 1.3, 2

def mailler(f, box, res, chunk=200000):
    """Champ implicite -> maillage triangulaire oriente vers l'exterieur."""
    lin = np.linspace(-box, box, res)
    X, Y, Z = np.meshgrid(lin, lin, lin, indexing='ij')
    pts = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    vals = np.empty(len(pts))
    for i in range(0, len(pts), chunk):
        vals[i:i+chunk] = np.asarray(f(pts[i:i+chunk])).ravel()
    vals = vals.reshape(res, res, res)
    pas = lin[1] - lin[0]
    v, fa, _, _ = measure.marching_cubes(vals, level=0.0, spacing=(pas, pas, pas))
    v = v + lin[0]
    v, fa = weld_vertices(v, fa)
    fa, vol, inv = orient_outward(v, fa)
    return v, fa, vertex_normals(v, fa), vol, inv

RES_OR = 160
V_or, F_or, N_or, vol_or, inv_or = mailler(champ, BOX, RES_OR)
topo_or = euler_characteristic(V_or, F_or)

print(f'ETALON — forme « {FORME} », grille {RES_OR}³')
print(f'  {len(V_or)} sommets, {len(F_or)} triangles')
print(f'  etanche = {is_watertight(V_or, F_or)}')
print(f'  chi     = {topo_or["chi"]}   (attendu {chi_attendu})   genre = {topo_or["genus"]:.1f}')
print(f'  aire    = {mesh_area(V_or, F_or):.5f}')
print(f'  volume  = {mesh_volume(V_or, F_or):.5f}   (volume signe {vol_or:+.5f}, inverse={inv_or})')

In [ ]:
# Boîte à outils graphique, réutilisée partout
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

plt.rcParams['figure.dpi'] = 110

def vue3d(ax, V, F, couleur='#4A90D9', alpha=0.95, titre='', box=None):
    box = box or BOX
    if V is not None and len(F):
        ax.add_collection3d(Poly3DCollection(V[F], facecolor=couleur,
                                             edgecolor='none', alpha=alpha))
    ax.set_xlim(-box, box); ax.set_ylim(-box, box); ax.set_zlim(-box, box)
    ax.set_box_aspect((1, 1, 1))
    ax.set_title(titre, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=24, azim=38)

def coupe(f, box, res=260, axe='z', niveau=0.0, chunk=200000):
    """Valeurs d'un champ sur une coupe plane, pour une carte de couleur."""
    lin = np.linspace(-box, box, res)
    A, B = np.meshgrid(lin, lin, indexing='ij')
    P = np.zeros((res * res, 3))
    i, j = {'z': (0, 1), 'y': (0, 2), 'x': (1, 2)}[axe]
    k = {'z': 2, 'y': 1, 'x': 0}[axe]
    P[:, i], P[:, j], P[:, k] = A.ravel(), B.ravel(), niveau
    out = np.empty(len(P))
    for s in range(0, len(P), chunk):
        out[s:s+chunk] = np.asarray(f(P[s:s+chunk])).ravel()
    return lin, out.reshape(res, res)

# FIGURE 1 — l'étalon
fig = plt.figure(figsize=(11, 4.2))
ax = fig.add_subplot(1, 2, 1, projection='3d')
vue3d(ax, V_or, F_or, titre=f'ÉTALON — « {FORME} »\nchi={topo_or["chi"]}, '
                            f'{len(F_or)} triangles')
ax2 = fig.add_subplot(1, 2, 2)
lin, Z = coupe(champ, BOX)
m = np.abs(Z).max()
im = ax2.imshow(Z.T, origin='lower', cmap='RdBu_r', extent=[-BOX, BOX, -BOX, BOX],
                vmin=-m, vmax=m)
ax2.contour(lin, lin, Z.T, levels=[0.0], colors='k', linewidths=1.6)
ax2.set_title('champ exact, coupe z = 0\n(trait noir : f = 0)', fontsize=10)
plt.colorbar(im, ax=ax2, shrink=0.85)
plt.tight_layout(); plt.show()

## 2) On abîme les données

Trois dégradations, cumulables, qui imitent un scanner réel :

- **`TROU_RAYON`** — on supprime tous les points du nuage situés dans une boule. C'est
  une **vue partielle** : une face que le capteur n'a pas atteinte.
- **`BRUIT`** — écart-type d'un bruit gaussien sur les **positions**.
- **`N_NUAGE`** — densité du nuage.

Les normales restent celles du maillage propre. C'est le **cas favorable** pour la
méthode du projet : sur de vraies données elles seraient réestimées à partir de points
bruités, donc elles seraient fausses elles aussi. On lui donne donc l'avantage.

In [ ]:
# ------------------------------------------------------------------
N_NUAGE    = 8000
TROU_RAYON = 0.80                     # 0 = pas de trou
TROU_CENTRE = np.array([0.0, -1.0, 0.0])
BRUIT      = 0.015                    # 0 = pas de bruit
# ------------------------------------------------------------------

rng = np.random.default_rng(0)
sel = rng.choice(len(V_or), size=min(N_NUAGE, len(V_or)), replace=False)
P_propre, N_propre = V_or[sel].copy(), N_or[sel].copy()

# 1. le trou
d_trou = np.linalg.norm(P_propre - TROU_CENTRE, axis=1)
garde = d_trou > TROU_RAYON
P_abime, N_abime = P_propre[garde].copy(), N_propre[garde].copy()
n_perdus = int((~garde).sum())

# 2. le bruit
if BRUIT > 0:
    P_abime = P_abime + BRUIT * rng.standard_normal(P_abime.shape)

nuage_propre = OrientedCloud(P_propre, N_propre)
nuage_abime  = OrientedCloud(P_abime,  N_abime)

print(f'nuage propre : {len(P_propre)} points')
print(f'trou         : rayon {TROU_RAYON} en {TROU_CENTRE} -> {n_perdus} points '
      f'supprimes ({100*n_perdus/len(P_propre):.1f} %)')
print(f'bruit        : sigma = {BRUIT}')
print(f'nuage abime  : {len(P_abime)} points')

esp_p = nuage_propre.tree.query(P_propre, k=2)[0][:, 1].mean()
esp_a = nuage_abime.tree.query(P_abime, k=2)[0][:, 1].mean()
print(f'espacement moyen : {esp_p:.4f} (propre) -> {esp_a:.4f} (abime)')

# FIGURE 2 — le nuage, avant et après
fig = plt.figure(figsize=(11, 4.2))
for k, (P, titre, coul) in enumerate([
        (P_propre, f'nuage PROPRE — {len(P_propre)} pts', '#2E8B57'),
        (P_abime,  f'nuage ABIMÉ — {len(P_abime)} pts\ntrou r={TROU_RAYON}, '
                   f'bruit σ={BRUIT}', '#C0392B')]):
    ax = fig.add_subplot(1, 2, k + 1, projection='3d')
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], s=1.2, c=coul, alpha=0.55)
    ax.set_xlim(-BOX, BOX); ax.set_ylim(-BOX, BOX); ax.set_zlim(-BOX, BOX)
    ax.set_box_aspect((1, 1, 1)); ax.set_title(titre, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.view_init(elev=24, azim=38)
plt.tight_layout(); plt.show()

## 3) La méthode du projet, sur le nuage abîmé

Signe par produit scalaire, `k = 5` voisins. On mesure **où** elle se trompe, en
séparant explicitement l'intérieur du trou du reste de l'espace.

In [ ]:
N_TEST = 40000
Q = rng.uniform(-BOX, BOX, size=(N_TEST, 3))
f_vrai = np.asarray(champ(Q)).ravel()
s_vrai = np.sign(f_vrai); s_vrai[s_vrai == 0] = 1.0

arbre_or = cKDTree(V_or)
dist_surf, _ = arbre_or.query(Q)          # distance vraie a la surface

s_proj = nuage_abime.sign(Q, k=5)

dans_trou = np.linalg.norm(Q - TROU_CENTRE, axis=1) < TROU_RAYON
zones = {
    'tout l espace':        np.ones(N_TEST, bool),
    'loin de la surface':   dist_surf > 0.05,
    'pres de la surface':   dist_surf <= 0.05,
    'DANS LE TROU':         dans_trou,
    'hors du trou':         ~dans_trou,
}
print('METHODE DU PROJET — accord du signe avec la verite (%)')
for nom, m in zones.items():
    if m.sum() == 0: continue
    print(f'  {nom:<22} n={int(m.sum()):>6}   {100*np.mean(s_proj[m]==s_vrai[m]):7.3f} %')

# FIGURE 3 — où sont les erreurs
faux = s_proj != s_vrai
fig = plt.figure(figsize=(11, 4.2))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(P_abime[:, 0], P_abime[:, 1], P_abime[:, 2], s=0.7, c='#BBBBBB', alpha=0.35)
if faux.any():
    ax.scatter(Q[faux, 0], Q[faux, 1], Q[faux, 2], s=6, c='#C0392B', alpha=0.9)
ax.set_xlim(-BOX, BOX); ax.set_ylim(-BOX, BOX); ax.set_zlim(-BOX, BOX)
ax.set_box_aspect((1, 1, 1)); ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
ax.view_init(elev=24, azim=38)
ax.set_title(f'erreurs de signe (rouge) — {int(faux.sum())} sur {N_TEST}\n'
             f'gris : le nuage abîmé', fontsize=10)

ax2 = fig.add_subplot(1, 2, 2)
bins = np.linspace(0, dist_surf.max(), 45)
ax2.hist(dist_surf[~faux], bins=bins, color='#2E8B57', alpha=0.65, label='correct')
ax2.hist(dist_surf[faux], bins=bins, color='#C0392B', alpha=0.85, label='erreur')
ax2.set_yscale('log'); ax2.set_xlabel('distance vraie à la surface')
ax2.set_ylabel('nombre de points (log)'); ax2.legend(fontsize=9)
ax2.set_title('les erreurs sont-elles collées à la surface,\nou dispersées ?',
              fontsize=10)
plt.tight_layout(); plt.show()

### Comment lire la figure 3

Sur un nuage **propre**, toutes les erreurs sont dans une pellicule collée à la surface —
l'histogramme rouge est écrasé contre la gauche.

Avec un trou, une deuxième population apparaît **loin** de la surface : ce sont les points
de l'intérieur du trou, où la méthode n'a plus aucun point de surface pertinent à
interroger. C'est le mode de défaillance que Points2Surf attaque.

### La figure de décision : à partir de quel trou la méthode lâche-t-elle ?

Celle-ci ne demande **aucun entraînement** — juste des KD-trees — donc elle est
instantanée. Et c'est probablement la figure la plus utile du notebook pour le rapport :
elle répond à *« à partir de quand a-t-on besoin d'autre chose ? »* par une courbe au
lieu d'une opinion.

In [ ]:
rayons = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
acc_global, acc_trou, frac_perdue = [], [], []

for R in rayons:
    g = np.linalg.norm(P_propre - TROU_CENTRE, axis=1) > R if R > 0 else np.ones(len(P_propre), bool)
    Pc_ = P_propre[g].copy()
    Nc_ = N_propre[g].copy()
    if BRUIT > 0:
        Pc_ = Pc_ + BRUIT * np.random.default_rng(7).standard_normal(Pc_.shape)
    nu = OrientedCloud(Pc_, Nc_)
    s = nu.sign(Q, k=5)
    acc_global.append(100 * np.mean(s == s_vrai))
    m = np.linalg.norm(Q - TROU_CENTRE, axis=1) < max(R, 1e-9)
    acc_trou.append(100 * np.mean(s[m] == s_vrai[m]) if m.sum() > 20 else np.nan)
    frac_perdue.append(100 * (1 - len(Pc_) / len(P_propre)))
    print(f'  rayon {R:.1f} : {len(Pc_):>6} pts ({frac_perdue[-1]:5.1f} % perdus)  '
          f'accord global {acc_global[-1]:6.2f} %  '
          f'dans le trou {acc_trou[-1]:6.2f} %')

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(rayons, acc_global, 'o-', color='#2E86C1', label='tout l espace')
ax.plot(rayons, acc_trou, 's--', color='#C0392B', label='dans le trou')
ax.set_xlabel('rayon du trou dans le nuage')
ax.set_ylabel('accord du signe (%)')
ax.set_title('Méthode du projet : jusqu où le trou est-il tolérable ?', fontsize=10)
ax.grid(alpha=0.3); ax.legend(fontsize=9)
ax2 = ax.twinx()
ax2.plot(rayons, frac_perdue, ':', color='#7F8C8D', lw=1.2)
ax2.set_ylabel('% de points perdus', color='#7F8C8D', fontsize=9)
plt.tight_layout(); plt.show()

print()
print('Lecture : tant que la courbe rouge reste haute, le produit scalaire suffit —')
print('inutile d aller chercher une methode apprise. Le point ou elle decroche est')
print('le critere de decision a reporter dans le rapport.')

## 4) Mini-Points2Surf

Deux branches, **aucune normale nulle part** :

| branche | entrée | sortie |
|---|---|---|
| distance | `K` plus proches points du nuage, recentrés et mis à l'échelle | `\|f\|` |
| signe | sous-échantillon **global** du nuage, vu depuis le point de requête | logit du signe |

L'encodeur est un PointNet minimal : un MLP appliqué à chaque point, puis un
**max-pooling**. Le pooling rend la sortie indépendante de l'**ordre** des points — la
propriété indispensable pour traiter un nuage, qui est un ensemble et non une liste.

In [ ]:
import torch.nn as nn

# N_GLOBAL = nombre de points que le reseau examine pour CHAQUE requete.
# ATTENTION : ce n'est PAS un simple curseur de vitesse. C'est ce qui donne
# au reseau sa VUE GLOBALE — donc la capacite meme qu'on cherche a tester.
# Mesure faite sur les deux valeurs, memes donnees, meme graine :
#     N_GLOBAL=512 -> chi=0 (juste), Chamfer 0.021, signe 93.5 % dans le trou
#     N_GLOBAL=128 -> chi=2 (FAUX),  Chamfer 0.057, signe 82.5 % dans le trou
# Le baisser va 4x plus vite et DETRUIT le resultat. Ne pas descendre sous 512.
SUR_GPU = (DEVICE.type == 'cuda')
if SUR_GPU:
    N_GLOBAL, LOT, EP_D, EP_S = 1024, 32768, 12, 25
else:
    torch.set_num_threads(os.cpu_count() or 4)   # torch se bride par defaut
    N_GLOBAL, LOT, EP_D, EP_S = 512, 4096, 10, 20
K_LOCAL = 32
print(f'materiel : {DEVICE}  |  N_GLOBAL={N_GLOBAL}  lot={LOT}  '
      f'epoques {EP_D}/{EP_S}  threads={torch.get_num_threads()}')

g_idx = rng.choice(len(P_abime), size=min(N_GLOBAL, len(P_abime)), replace=False)
VUE_GLOBALE = P_abime[g_idx]

def rustine(Q, K=K_LOCAL):
    Q = np.asarray(Q, dtype=np.float64).reshape(-1, 3)
    _, idx = nuage_abime.tree.query(Q, k=K)
    p = P_abime[idx] - Q[:, None, :]
    ech = np.maximum(np.linalg.norm(p, axis=2).max(axis=1), 1e-9)
    return p / ech[:, None, None], ech

def vue(Q):
    Q = np.asarray(Q, dtype=np.float64).reshape(-1, 3)
    return (VUE_GLOBALE[None] - Q[:, None, :]) / BOX

class PointNet(nn.Module):
    def __init__(self, d_out=128, h=64):
        super().__init__()
        self.pp = nn.Sequential(nn.Linear(3, h), nn.ReLU(),
                                nn.Linear(h, 2*h), nn.ReLU())
        self.tete = nn.Sequential(nn.Linear(2*h, d_out), nn.ReLU(),
                                  nn.Linear(d_out, d_out), nn.ReLU(),
                                  nn.Linear(d_out, 1))
    def forward(self, x):
        return self.tete(self.pp(x).max(dim=1).values).squeeze(-1)

net_d = PointNet().to(DEVICE)     # rustine locale -> |f|
net_s = PointNet().to(DEVICE)     # vue globale    -> signe
print('branche distance :', sum(p.numel() for p in net_d.parameters()), 'poids')
print('branche signe    :', sum(p.numel() for p in net_s.parameters()), 'poids')

In [ ]:
# Jeu d'entraînement : la supervision vient de l'ÉTALON, comme dans l'article
def requetes(n, seed, part_proche=0.5, sigma=0.06):
    r = np.random.default_rng(seed)
    n_p = int(n * part_proche)
    base = P_abime[r.choice(len(P_abime), size=n_p, replace=True)]
    return np.clip(np.vstack([base + sigma * r.standard_normal((n_p, 3)),
                              r.uniform(-BOX, BOX, size=(n - n_p, 3))]), -BOX, BOX)

def cibles(Q):
    d, _ = arbre_or.query(Q)                      # distance a l'ETALON
    s = np.sign(np.asarray(champ(Q)).ravel()); s[s == 0] = 1.0
    return d, s

Q_tr = requetes(60000, seed=1); d_tr, s_tr = cibles(Q_tr)
Q_va = requetes(15000, seed=2); d_va, s_va = cibles(Q_va)

def preparer(Q, d):
    p, ech = rustine(Q)
    return (torch.tensor(p, dtype=torch.float32),
            torch.tensor(vue(Q), dtype=torch.float32),
            torch.tensor(d / ech, dtype=torch.float32),
            torch.tensor(ech, dtype=torch.float32))

P_tr, G_tr, dn_tr, e_tr = preparer(Q_tr, d_tr)
P_va, G_va, dn_va, e_va = preparer(Q_va, d_va)
y_tr = torch.tensor((s_tr > 0).astype(np.float32))
y_va = torch.tensor((s_va > 0).astype(np.float32))
print(f'entrainement : {len(Q_tr)} requetes | dedans {int((s_tr<0).sum())} '
      f'/ dehors {int((s_tr>0).sum())}')

### Entraînement, avec la courbe qui se dessine en direct

La cellule redessine la perte toutes les quelques époques : tu vois l'apprentissage se
faire. La courbe de validation permet de repérer un surapprentissage.

In [ ]:
from IPython.display import clear_output

# Tout le jeu part sur le GPU UNE fois. Sur T4, transferer de petits lots a
# chaque pas coute plus cher que le calcul lui-meme : la carte attend le bus.
# Avec les donnees deja sur place et de gros lots, on sature les coeurs CUDA
# et il faut beaucoup moins d'epoques pour le meme travail.
def sur_device(*ts):
    return [t.to(DEVICE, non_blocking=True) for t in ts]

def memoire():
    if DEVICE.type != 'cuda':
        return ''
    a = torch.cuda.memory_allocated() / 2**30
    r = torch.cuda.get_device_properties(0).total_memory / 2**30
    return f' | VRAM {a:.2f}/{r:.1f} Go'

def entrainer(net, X, y, Xv, yv, n_epochs, perte='l1', bs=None, lr=2e-3,
              nom='', chaque=2):
    bs = bs or LOT
    X, y, Xv, yv = sur_device(X, y, Xv, yv)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    fn = nn.L1Loss() if perte == 'l1' else nn.BCEWithLogitsLoss()
    n = len(X)
    h_tr, h_va = [], []
    for ep in range(n_epochs):
        net.train()
        perm = torch.randperm(n, device=DEVICE)      # melange sur le GPU aussi
        tot = torch.zeros((), device=DEVICE)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            loss = fn(net(X[idx]), y[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.detach() * len(idx)
        sched.step()
        h_tr.append(float(tot) / n)
        net.eval()
        with torch.no_grad():
            h_va.append(float(fn(net(Xv), yv)))
        if ep % chaque == 0 or ep == n_epochs - 1:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 2.8))
            ax.plot(h_tr, label='entraînement', color='#2E86C1')
            ax.plot(h_va, label='validation', color='#C0392B', ls='--')
            ax.set_xlabel('époque'); ax.set_ylabel(perte.upper())
            ax.set_title(f'branche {nom} — époque {ep+1}/{n_epochs}  '
                         f'(train {h_tr[-1]:.4f} · val {h_va[-1]:.4f})'
                         + memoire(), fontsize=10)
            ax.legend(fontsize=9); ax.grid(alpha=0.25)
            plt.tight_layout(); plt.show()
    return h_tr, h_va

h_d = entrainer(net_d, P_tr, dn_tr, P_va, dn_va, n_epochs=EP_D,
                perte='l1', nom='DISTANCE (rustine locale)', chaque=1)

In [ ]:
# La branche signe est la tache difficile : elle merite plus de passes,
# mais avec de gros lots sur GPU elles coutent quelques secondes chacune.
h_s = entrainer(net_s, G_tr, y_tr, G_va, y_va, n_epochs=EP_S,
                perte='bce', nom='SIGNE (vue globale)', chaque=2)

## 5) Ce que chaque branche a appris

Deux cartes en coupe : la distance prédite face à la vraie, et la **frontière de
décision** de la branche signe. C'est là qu'on voit concrètement si le réseau a
« deviné » le trou.

In [ ]:
@torch.no_grad()
def predire(net, X, bs=8192):
    net.eval()
    return torch.cat([net(X[i:i+bs].to(DEVICE)).cpu()
                      for i in range(0, len(X), bs)]).numpy()

# qualité chiffrée
d_pred = predire(net_d, P_va) * e_va.numpy()
err_d = np.abs(d_pred - d_va)
print('BRANCHE DISTANCE (locale, sans normales)')
print(f'  erreur absolue moyenne : {err_d.mean():.5f}')
print(f'  erreur mediane         : {np.median(err_d):.5f}')

s_p2s_va = np.where(predire(net_s, G_va) > 0, 1.0, -1.0)
s_proj_va = nuage_abime.sign(Q_va, k=5)
d_va_surf, _ = arbre_or.query(Q_va)
trou_va = np.linalg.norm(Q_va - TROU_CENTRE, axis=1) < TROU_RAYON

print()
print('BRANCHE SIGNE — accord avec la verite (%)')
print(f"  {'zone':<24}{'n':>7}{'P2S global':>13}{'projet local':>14}")
print('  ' + '-' * 58)
for nom, m in (('tout l espace', np.ones(len(Q_va), bool)),
               ('loin de la surface', d_va_surf > 0.05),
               ('pres de la surface', d_va_surf <= 0.05),
               ('DANS LE TROU', trou_va),
               ('hors du trou', ~trou_va)):
    if m.sum() == 0: continue
    a = 100*np.mean(s_p2s_va[m] == s_va[m]); b = 100*np.mean(s_proj_va[m] == s_va[m])
    print(f'  {nom:<24}{int(m.sum()):>7}{a:>12.3f}%{b:>13.3f}%')

In [ ]:
# FIGURE 4 — les deux branches en coupe
RES_C = 150
lin_c = np.linspace(-BOX, BOX, RES_C)
A, B = np.meshgrid(lin_c, lin_c, indexing='ij')
Pc = np.column_stack([A.ravel(), B.ravel(), np.zeros(A.size)])

d_vrai_c, _ = arbre_or.query(Pc)
pat_c, ech_c = rustine(Pc)
d_pred_c = predire(net_d, torch.tensor(pat_c, dtype=torch.float32)) * ech_c
log_c = predire(net_s, torch.tensor(vue(Pc), dtype=torch.float32))
s_proj_c = nuage_abime.sign(Pc, k=5)
f_vrai_c = np.asarray(champ(Pc)).ravel()

def cercle_trou(ax):
    d0 = TROU_RAYON**2 - TROU_CENTRE[2]**2
    if d0 > 0:
        t = np.linspace(0, 2*np.pi, 200)
        ax.plot(TROU_CENTRE[0] + np.sqrt(d0)*np.cos(t),
                TROU_CENTRE[1] + np.sqrt(d0)*np.sin(t),
                color='#F39C12', lw=2, ls='--')

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
ext = [-BOX, BOX, -BOX, BOX]
for ax, Z, titre, cm in [
        (axes[0], d_vrai_c, 'distance VRAIE à l étalon', 'viridis'),
        (axes[1], d_pred_c, 'distance PRÉDITE (rustine locale)', 'viridis')]:
    im = ax.imshow(Z.reshape(RES_C, RES_C).T, origin='lower', extent=ext, cmap=cm)
    ax.contour(lin_c, lin_c, f_vrai_c.reshape(RES_C, RES_C).T, levels=[0],
               colors='w', linewidths=1.4)
    cercle_trou(ax); ax.set_title(titre, fontsize=9.5)
    plt.colorbar(im, ax=ax, shrink=0.8)

im = axes[2].imshow(np.tanh(log_c).reshape(RES_C, RES_C).T, origin='lower',
                    extent=ext, cmap='coolwarm', vmin=-1, vmax=1)
axes[2].contour(lin_c, lin_c, log_c.reshape(RES_C, RES_C).T, levels=[0],
                colors='k', linewidths=1.6)
axes[2].contour(lin_c, lin_c, f_vrai_c.reshape(RES_C, RES_C).T, levels=[0],
                colors='w', linewidths=1.2, linestyles='--')
cercle_trou(axes[2])
axes[2].set_title('SIGNE appris (vue globale)\nnoir : sa frontière · blanc : la vraie',
                  fontsize=9.5)
plt.colorbar(im, ax=axes[2], shrink=0.8)

axes[3].imshow(s_proj_c.reshape(RES_C, RES_C).T, origin='lower', extent=ext,
               cmap='coolwarm', vmin=-1, vmax=1)
axes[3].contour(lin_c, lin_c, f_vrai_c.reshape(RES_C, RES_C).T, levels=[0],
                colors='w', linewidths=1.2, linestyles='--')
cercle_trou(axes[3])
axes[3].set_title('SIGNE du projet (produit scalaire)\nblanc : la vraie surface',
                  fontsize=9.5)
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print('Le cercle orange marque la coupe du trou. Regarde qui reste cohérent dedans.')

## 6) Les deux reconstructions, face à l'étalon

On assemble `f = signe × |f|` pour Points2Surf, et pour la méthode du projet on
utilise sa pseudo-SDF (`signe × distance au nuage`). Puis marching cubes sur les deux,
et comparaison à l'étalon.

In [ ]:
RES_G = 96
lin_g = np.linspace(-BOX, BOX, RES_G)
Xg, Yg, Zg = np.meshgrid(lin_g, lin_g, lin_g, indexing='ij')
GRILLE = np.column_stack([Xg.ravel(), Yg.ravel(), Zg.ravel()])
print(f'grille {RES_G}³ = {len(GRILLE)} points')

def sdf_p2s(Q, bs=40000):
    out = np.empty(len(Q))
    for i in range(0, len(Q), bs):
        q = Q[i:i+bs]
        p, ech = rustine(q)
        d = predire(net_d, torch.tensor(p, dtype=torch.float32)) * ech
        s = np.where(predire(net_s, torch.tensor(vue(q), dtype=torch.float32)) > 0, 1., -1.)
        out[i:i+bs] = s * np.abs(d)
    return out

def sdf_projet(Q, bs=200000):
    out = np.empty(len(Q))
    for i in range(0, len(Q), bs):
        q = Q[i:i+bs]
        out[i:i+bs] = nuage_abime.sign(q, k=5) * nuage_abime.distance(q)
    return out

def reconstruire(vals, nom):
    vmin, vmax = float(vals.min()), float(vals.max())
    if not (vmin < 0.0 < vmax):
        print(f'  [{nom}] f ne change pas de signe ([{vmin:+.3f},{vmax:+.3f}]) '
              f'-> aucune surface')
        return None, None, None, None
    pas = lin_g[1] - lin_g[0]
    v, f, _, _ = measure.marching_cubes(vals.reshape(RES_G, RES_G, RES_G),
                                        level=0.0, spacing=(pas, pas, pas))
    v = v + lin_g[0]
    v, f = weld_vertices(v, f)
    t = euler_characteristic(v, f)
    c = chamfer_distance(v, V_or)
    print(f'  [{nom}] {len(v)} sommets · chi={t["chi"]} · aire={mesh_area(v,f):.4f} · '
          f'volume={mesh_volume(v,f):.4f} · Chamfer={c["chamfer"]:.5f}')
    return v, f, t, c

print('reconstruction...')
V_p2s, F_p2s, T_p2s, C_p2s = reconstruire(sdf_p2s(GRILLE), 'Points2Surf')
V_prj, F_prj, T_prj, C_prj = reconstruire(sdf_projet(GRILLE), 'projet')

In [ ]:
# FIGURE 5 — les trois surfaces côte à côte
fig = plt.figure(figsize=(15, 4.6))
panneaux = [
    (V_or, F_or, '#2E8B57', f'ÉTALON\nchi={topo_or["chi"]}, aire={mesh_area(V_or,F_or):.3f}'),
    (V_prj, F_prj, '#2E86C1',
     'PROJET (produit scalaire)\n' + (f'chi={T_prj["chi"]}, Chamfer={C_prj["chamfer"]:.4f}'
                                       if T_prj else 'aucune surface')),
    (V_p2s, F_p2s, '#C0392B',
     'MINI-POINTS2SURF (sans normales)\n' + (f'chi={T_p2s["chi"]}, Chamfer={C_p2s["chamfer"]:.4f}'
                                              if T_p2s else 'aucune surface')),
]
for k, (V, F, c, t) in enumerate(panneaux):
    ax = fig.add_subplot(1, 3, k + 1, projection='3d')
    vue3d(ax, V, F, couleur=c, titre=t)
plt.tight_layout(); plt.show()

### Version explorable à la souris

La même chose en interactif : clic-glisser pour tourner, molette pour zoomer,
et **clic sur la légende** pour afficher ou masquer chaque objet.

L'expérience à faire : affiche l'ÉTALON **et** la méthode du projet en même temps,
puis tourne jusqu'à la zone effacée (vers `y = −1`). Tu verras de tes yeux ce que
`chi = −1` veut dire — la reconstruction se déforme précisément là où il manque
des points. Ajoute les points rouges pour voir la boule qui a été gommée.

In [ ]:
# FIGURE 6 — les surfaces explorables (plotly)
import plotly.graph_objects as go
import plotly.io as pio

# "plotly_mimetype" pour VS Code, "notebook" embarque la librairie dans le
# fichier : la figure reste manipulable meme hors ligne, apres reouverture.
pio.renderers.default = "plotly_mimetype+notebook"

def maillage3d(V, F, nom, couleur, visible=True, opacite=1.0):
    return go.Mesh3d(
        x=V[:, 0], y=V[:, 1], z=V[:, 2],
        i=F[:, 0], j=F[:, 1], k=F[:, 2],
        color=couleur, opacity=opacite, name=nom, showlegend=True,
        visible=visible, hoverinfo="name", flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.85, specular=0.25, roughness=0.55),
        lightposition=dict(x=120, y=180, z=140))

fig3d = go.Figure()
fig3d.add_trace(maillage3d(V_or, F_or, f"ÉTALON (chi={topo_or['chi']})", "#2E8B57"))

if V_prj is not None:
    fig3d.add_trace(maillage3d(V_prj, F_prj,
                    f"méthode du projet (chi={T_prj['chi']})", "#2E86C1",
                    visible="legendonly"))
if V_p2s is not None:
    fig3d.add_trace(maillage3d(V_p2s, F_p2s,
                    f"mini-Points2Surf (chi={T_p2s['chi']})", "#C0392B",
                    visible="legendonly"))

fig3d.add_trace(go.Scatter3d(
    x=P_abime[:, 0], y=P_abime[:, 1], z=P_abime[:, 2], mode="markers",
    marker=dict(size=1.4, color="#7F8C8D"), name="nuage abîmé",
    visible="legendonly", hoverinfo="name"))

efface = P_propre[~garde]
fig3d.add_trace(go.Scatter3d(
    x=efface[:, 0], y=efface[:, 1], z=efface[:, 2], mode="markers",
    marker=dict(size=1.8, color="#E67E22"), name="points EFFACÉS",
    visible="legendonly", hoverinfo="name"))

fig3d.update_layout(
    title=dict(text="Clique sur la légende pour afficher / masquer",
               x=0.5, font=dict(size=14)),
    scene=dict(xaxis=dict(range=[-BOX, BOX], title="x"),
               yaxis=dict(range=[-BOX, BOX], title="y"),
               zaxis=dict(range=[-BOX, BOX], title="z"),
               aspectmode="cube",
               camera=dict(eye=dict(x=1.5, y=1.5, z=1.1))),
    legend=dict(itemsizing="constant", x=0.01, y=0.99,
                bgcolor="rgba(255,255,255,0.75)"),
    margin=dict(l=0, r=0, t=46, b=0), height=700)

fig3d.show()

# Copie autonome, ouvrable dans un navigateur et envoyable par mail
import os
os.makedirs("output", exist_ok=True)
fig3d.write_html("output/surfaces_3d.html", include_plotlyjs=True, full_html=True)
print("copie autonome ecrite : output/surfaces_3d.html")

In [ ]:
# VERDICT
print('=' * 68)
print(f'VERDICT — forme « {FORME} », trou r={TROU_RAYON}, bruit σ={BRUIT}')
print('=' * 68)
print(f"  {'':<26}{'ÉTALON':>12}{'projet':>13}{'Points2Surf':>14}")
print('  ' + '-' * 64)
def ligne(nom, a, b, c, fmt='{:.4f}'):
    fb = fmt.format(b) if b is not None else 'n/a'
    fc = fmt.format(c) if c is not None else 'n/a'
    print(f'  {nom:<26}{fmt.format(a):>12}{fb:>13}{fc:>14}')

ligne('chi (topologie)', topo_or['chi'],
      T_prj['chi'] if T_prj else None, T_p2s['chi'] if T_p2s else None, '{:.0f}')
ligne('aire', mesh_area(V_or, F_or),
      mesh_area(V_prj, F_prj) if V_prj is not None else None,
      mesh_area(V_p2s, F_p2s) if V_p2s is not None else None)
ligne('volume', mesh_volume(V_or, F_or),
      mesh_volume(V_prj, F_prj) if V_prj is not None else None,
      mesh_volume(V_p2s, F_p2s) if V_p2s is not None else None)
print()
print(f"  Chamfer à l'étalon        projet = "
      f"{C_prj['chamfer']:.5f}" if C_prj else '  projet : aucune surface')
print(f"                            P2S    = "
      f"{C_p2s['chamfer']:.5f}" if C_p2s else '  P2S : aucune surface')
print(f'  maille de la grille       {2*BOX/RES_G:.5f}')
print()
print('  chi est le juge le plus severe : il ne se rattrape pas. Une surface')
print('  percee, doublee ou bouchee se voit immediatement dessus.')

## 7) Comment lire tout ça

**Sur la branche distance.** Compare son erreur à la maille de la grille. Si elle est
nettement en dessous, la thèse de l'article est confirmée : `|f|` s'estime localement,
même sur un nuage abîmé. C'est le morceau facile — et il le reste.

**Sur la branche signe, hors du trou.** Le produit scalaire du projet devrait gagner : il
est *exact* là où il dispose de points de surface, tandis que le réseau *approxime*. Un
réseau ne bat pas un calcul exact quand celui-ci est applicable.

**Sur la branche signe, DANS le trou.** C'est le cœur de l'expérience. La méthode du
projet n'a plus rien à interroger : son plus proche voisin est au bord du trou, et sa
normale ne décrit plus la géométrie locale. Le réseau, lui, voit la forme entière et peut
extrapoler. Si l'écart se creuse ici en sa faveur, **la thèse de Points2Surf est vérifiée
sur vos données** — et vous savez exactement dans quel régime elle s'applique.

**Sur `χ`.** C'est le juge sans appel. Sur le tore, `χ = 0` : une reconstruction qui
bouche le trou central donnera `χ = 2` et se trahira, même avec un excellent Chamfer.
C'est précisément pourquoi Nathan a écrit `invariants.py`.

## Ce que ça vous donne pour le rapport

Une réponse chiffrée à la question « faut-il aller vers les méthodes apprises ? » :

> Tant que le maillage ÉF est complet, la méthode du produit scalaire suffit et fait
> mieux. Dès qu'il manque de la matière, elle s'effondre là où il en manque, et c'est
> à ce moment-là qu'un *prior* appris devient nécessaire.

C'est un critère de décision, pas une opinion. Et il se teste sur les vraies données dès
leur arrivée avec `invariants.is_watertight`.

## À faire varier

- `TROU_RAYON = 0` → retour au cas propre, où les deux méthodes font jeu égal.
- Agrandir le trou progressivement (0.3, 0.5, 0.8) et tracer les deux courbes de
  précision : c'est **la** figure à mettre dans le rapport.
- `FORME = 'tore'` puis `'tore_sphere'` : l'arête vive de l'union est un second type de
  difficulté, indépendant du trou.
- `BRUIT` seul, sans trou : sépare l'effet du bruit de celui de la vue partielle.

## Limites assumées

1. **Une seule forme à la fois.** Le vrai Points2Surf tire sa force d'un entraînement sur
   des milliers d'objets. Ici on entraîne et on évalue sur la même forme : les scores
   sont optimistes et ne mesurent **pas** la généralisation, qui est pourtant l'argument
   central de l'article.
2. **Les normales du nuage abîmé restent celles du maillage propre** — un avantage
   accordé à la méthode du projet, qui sur de vraies données ne l'aurait pas.
3. **Sous-échantillon global fixe**, au lieu du gradient de densité de l'article.